# load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import joblib

print("Libraries imported successfully")

Libraries imported successfully


In [ ]:
file_path =  '/content/drive/MyDrive/rideflow_AI/processed_data/cleaned_data.csv'
data = pd.read_csv(file_path)
data

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/rideflow_AI/processed_data/cleaned_data.csv'

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2160266 entries, 0 to 2160265
Data columns (total 29 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0   ride_id                       int64  
 1   Booking_time                  object 
 2   pickup_zone                   object 
 3   drop_zone                     object 
 4   driver_id                     int64  
 5   customer_id                   int64  
 6   trip_km                       float64
 7   base_fare                     float64
 8   surge_multiplier              float64
 9   total_fare                    float64
 10  customer_rating               float64
 11  estimated_trip_duration_min   int64  
 12  actual_trip_duration_min      int64  
 13  estimated_driver_arrival_min  int64  
 14  ride_status                   object 
 15  reason_cancellation           object 
 16  traffic_level                 object 
 17  weather                       object 
 18  driver_status         

# ETA prediction

In [ ]:
data['Booking_time'] = pd.to_datetime(data['Booking_time'])
data['hour_bucket'] = data['Booking_time'].dt.floor('h')

df_eta = data[['hour_bucket','pickup_zone', 'drop_zone','estimated_driver_arrival_min',
               'trip_km' ,'estimated_trip_duration_min', 'actual_trip_duration_min','traffic_level_enc',
       'weather_enc', 'pickup_zone_enc','drop_zone_enc']].copy()

In [ ]:
df_eta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2160266 entries, 0 to 2160265
Data columns (total 11 columns):
 #   Column                        Dtype         
---  ------                        -----         
 0   hour_bucket                   datetime64[ns]
 1   pickup_zone                   object        
 2   drop_zone                     object        
 3   estimated_driver_arrival_min  int64         
 4   trip_km                       float64       
 5   estimated_trip_duration_min   int64         
 6   actual_trip_duration_min      int64         
 7   traffic_level_enc             int64         
 8   weather_enc                   int64         
 9   pickup_zone_enc               int64         
 10  drop_zone_enc                 int64         
dtypes: datetime64[ns](1), float64(1), int64(7), object(2)
memory usage: 181.3+ MB


In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
df_eta["hour_bucket"] = df_eta["hour_bucket"].dt.floor("h")
df_eta['hour'] = df_eta['hour_bucket'].dt.hour
df_eta['day'] = df_eta['hour_bucket'].dt.day
df_eta['month'] = df_eta['hour_bucket'].dt.month
df_eta['is_weekday'] = df_eta['hour_bucket'].dt.weekday
df_eta["is_weekend"] = df_eta["day"].apply(lambda x: 1 if x >= 5 else 0)


eta_features = [
    'pickup_zone_enc',
    'drop_zone_enc',
    'weather_enc',
    'traffic_level_enc',
    'hour',
    'is_weekend'
]

eta_target = 'estimated_trip_duration_min'

df_eta = df_eta.sort_values('hour_bucket')

# Sort chronologically
df_eta = df_eta.sort_values('hour_bucket').reset_index(drop=True)

# 2. Use your Strict Chronological Split
split_time = pd.Timestamp("2026-03-15 09:00:00")
train_mask = df_eta["hour_bucket"] < split_time
test_mask = df_eta["hour_bucket"] >= split_time

train_df = df_eta[train_mask]
test_df = df_eta[test_mask]

X_train_raw = train_df[eta_features].values
y_train_raw = train_df[eta_target].values
X_test_raw = test_df[eta_features].values
y_test_raw = test_df[eta_target].values

print(f"Training samples: {len(X_train_raw):,}, Test samples: {len(X_test_raw):,}")

# 3. Fit Scalers ONLY on Train Data (Prevents Data Leakage)
scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_train_e = scaler_x.fit_transform(X_train_raw)
X_test_e = scaler_x.transform(X_test_raw)  # Transform only!

y_train_e = scaler_y.fit_transform(y_train_raw.reshape(-1, 1))
y_test_e = scaler_y.transform(y_test_raw.reshape(-1, 1))

# 4. Reshape into 3D for LSTM: (samples, timesteps, features)
X_train_3d = X_train_e.reshape((X_train_e.shape[0], 1, X_train_e.shape[1]))
X_test_3d = X_test_e.reshape((X_test_e.shape[0], 1, X_test_e.shape[1]))

Training samples: 1,753,887, Test samples: 406,379


In [ ]:
eta_model = Sequential([
    LSTM(64, activation='relu', input_shape=(1, X_train_3d.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)  # Single output for regression
])

eta_model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

# 6. Crash-Prevention: Early Stopping & Larger Batch Size
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = eta_model.fit(
    X_train_3d, y_train_e,
    epochs=10,              # Kept low (10) because 2.1M rows learn fast per epoch
    batch_size=256,         # Larger batch size prevents memory crashes
    validation_split=0.1,   # 10% validation from training data
    callbacks=[early_stopping],
    verbose=1
)

# 7. Evaluate on Test Set
loss, mae = eta_model.evaluate(X_test_3d, y_test_e)
print(f"\nTest Loss (MSE): {loss:.4f}")
print(f"Test MAE: {mae:.4f}")

# 8. Make Predictions & Inverse Transform back to Minutes
y_pred_scaled = eta_model.predict(X_test_3d)
y_pred_minutes = scaler_y.inverse_transform(y_pred_scaled)

print("\nSample Predictions (Minutes):")
print(y_pred_minutes[:5].flatten())

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
6167/6167 ━━━━━━━━━━━━━━━━━━━━ 42s 6ms/step - loss: 0.3790 - mae: 0.4712 - val_loss: 0.1454 - val_mae: 0.3144
Epoch 2/10
6167/6167 ━━━━━━━━━━━━━━━━━━━━ 40s 6ms/step - loss: 0.1426 - mae: 0.2923 - val_loss: 0.1024 - val_mae: 0.2322
Epoch 3/10
6167/6167 ━━━━━━━━━━━━━━━━━━━━ 51s 8ms/step - loss: 0.0816 - mae: 0.2080 - val_loss: 0.0690 - val_mae: 0.1955
Epoch 4/10
6167/6167 ━━━━━━━━━━━━━━━━━━━━ 73s 7ms/step - loss: 0.0554 - mae: 0.1647 - val_loss: 0.0618 - val_mae: 0.1805
Epoch 5/10
6167/6167 ━━━━━━━━━━━━━━━━━━━━ 36s 6ms/step - loss: 0.0441 - mae: 0.1430 - val_loss: 0.0528 - val_mae: 0.1611
Epoch 6/10
6167/6167 ━━━━━━━━━━━━━━━━━━━━ 56s 8ms/step - loss: 0.0375 - mae: 0.1294 - val_loss: 0.0638 - val_mae: 0.1707
Epoch 7/10
6167/6167 ━━━━━━━━━━━━━━━━━━━━ 44s 7ms/step - loss: 0.0326 - mae: 0.1199 - val_loss: 0.0503 - val_mae: 0.1547
Epoch 8/10
6167/6167 ━━━━━━━━━━━━━━━━━━━━ 36s 6ms/step - loss: 0.0282 - mae: 0.1111 - val_loss: 0.0510 - val_mae: 0.1604
Epoch 9/10
6167/6167 ━━━━━━━━━━━

In [ ]:
# Calculate RMSE in original minutes scale
rmse_eta = np.sqrt(mean_squared_error(y_test_raw, y_pred_minutes))

print(f"Test RMSE: {rmse_eta:.2f} minutes")


Test RMSE: 2.97 minutes


In [ ]:
baseline_pred = np.repeat(
    y_test_raw.mean(),
    len(y_test_raw))

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test_raw,
        baseline_pred   ))

print("Baseline RMSE:", baseline_rmse)
print("LSTM RMSE:", rmse_eta)

improvement = (
    (baseline_rmse - rmse_eta)
    / baseline_rmse
) * 100

print("Improvement (%):", improvement)

Baseline RMSE: 14.819982174952699
LSTM RMSE: 2.9663810513912776
Improvement (%): 79.98390945162696


In [ ]:
# 1. Scale and reshape full features
X_full_scaled = scaler_x.transform(df_eta[eta_features])
X_full_3d = X_full_scaled.reshape((X_full_scaled.shape[0], 1, X_full_scaled.shape[1]))

# 2. Predict and inverse transform
scaled_preds = eta_model.predict(X_full_3d)
df_eta['predicted_trip_time'] = scaler_y.inverse_transform(scaled_preds).flatten()
df_eta['predicted_trip_time'] = df_eta['predicted_trip_time'].round().astype(int)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


67509/67509 ━━━━━━━━━━━━━━━━━━━━ 108s 2ms/step


# demand hotspot model

In [ ]:
import pandas as pd

df_hotspot = data.copy()

# Time feature extractions
df_hotspot['hour_bucket'] = pd.to_datetime(df_hotspot['Booking_time']).dt.floor('h')
df_hotspot['hour'] = df_hotspot['hour_bucket'].dt.hour
df_hotspot['day'] = df_hotspot['hour_bucket'].dt.day
df_hotspot['month'] = df_hotspot['hour_bucket'].dt.month
df_hotspot['is_weekday'] = df_hotspot['hour_bucket'].dt.weekday
df_hotspot["is_weekend"] = df_hotspot["day"].apply(lambda x: 1 if x >= 5 else 0)

# Aggregate to get ride demand per zone and hour bucket
df_hotspot = df_hotspot.groupby(
    ['pickup_zone_enc', 'hour_bucket', 'is_weekday', 'is_weekend']
).size().reset_index(name='ride_demand')

# Create Hotspot target labels
df_hotspot['hotspot'] = pd.qcut(
    df_hotspot['ride_demand'],
    q=3,
    labels=['Low', 'Medium', 'High']
)

# Extract hour/day/month again from hour_bucket if needed
df_hotspot['hour'] = df_hotspot['hour_bucket'].dt.hour
df_hotspot['day'] = df_hotspot['hour_bucket'].dt.day
df_hotspot['month'] = df_hotspot['hour_bucket'].dt.month

# 1. Encode hotspot column and explicitly cast to integer
bins = {'Low': 0, 'Medium': 1, 'High': 2}
df_hotspot['hotspot_enc'] = df_hotspot['hotspot'].map(bins).astype(int)


# Corrected feature list
hotspot_features = [
    'pickup_zone_enc',
    'hour',
    'is_weekday',
    'is_weekend'
]

hotspot_target = 'hotspot_enc'

# Sort chronologically and reset index
df_hotspot = df_hotspot.sort_values('hour_bucket').reset_index(drop=True)

# Use strict chronological split (Fixed df_eta -> df_hotspot)
split_time = pd.Timestamp("2026-03-15 09:00:00")
train_mask = df_hotspot["hour_bucket"] < split_time
test_mask = df_hotspot["hour_bucket"] >= split_time

hotspot_x = df_hotspot[hotspot_features]
hotspot_y = df_hotspot[hotspot_target]

hotspot_x_train = hotspot_x.loc[train_mask]
hotspot_x_test = hotspot_x.loc[test_mask]

hotspot_y_train = hotspot_y.loc[train_mask]
hotspot_y_test = hotspot_y.loc[test_mask]

scaler_hotspot = StandardScaler()
hotspot_x_train_scaled = scaler_hotspot.fit_transform(hotspot_x_train)
hotspot_x_test_scaled = scaler_hotspot.transform(hotspot_x_test)

print("Training ends:", df_hotspot.loc[train_mask, "hour_bucket"].max())
print("Testing starts:", df_hotspot.loc[test_mask, "hour_bucket"].min())

print('x_train shape:', hotspot_x_train.shape)
print('y_train shape:', hotspot_y_train.shape)
print('x_test shape:', hotspot_x_test.shape)
print('y_test shape:', hotspot_y_test.shape)

Training ends: 2026-03-15 08:00:00
Testing starts: 2026-03-15 09:00:00
x_train shape: (17610, 4)
y_train shape: (17610,)
x_test shape: (3990, 4)
y_test shape: (3990,)


In [ ]:
 from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

hotspot_model = Sequential([
    Dense(64, activation='relu', input_shape=(hotspot_x_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(3, activation='softmax') ])

hotspot_model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = hotspot_model.fit(
    hotspot_x_train_scaled, hotspot_y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

# 5. Evaluate on Test Set
loss, accuracy = hotspot_model.evaluate(hotspot_x_test_scaled, hotspot_y_test)
print(f"\nTest Accuracy: {accuracy:.4f}")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
221/221 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.5717 - loss: 0.8908 - val_accuracy: 0.6488 - val_loss: 0.7540
Epoch 2/20
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6345 - loss: 0.7535 - val_accuracy: 0.6661 - val_loss: 0.6928
Epoch 3/20
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6494 - loss: 0.7138 - val_accuracy: 0.6726 - val_loss: 0.6629
Epoch 4/20
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6591 - loss: 0.6868 - val_accuracy: 0.6942 - val_loss: 0.6399
Epoch 5/20
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6795 - loss: 0.6632 - val_accuracy: 0.7212 - val_loss: 0.6166
Epoch 6/20
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6970 - loss: 0.6408 - val_accuracy: 0.7513 - val_loss: 0.5934
Epoch 7/20
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7164 - loss: 0.6167 - val_accuracy: 0.7578 - val_loss: 0.5700
Epoch 8/20
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7271 - loss: 0.6002 - val_accuracy: 0.

In [ ]:
import numpy as np
import pandas as pd

# 1. Create a direct mapping dictionary from encoded numbers to real zone names
# (Assuming your raw data has 'pickup_zone' and you previously created 'pickup_zone_enc')
if 'pickup_zone' in data.columns and 'pickup_zone_enc' in data.columns:
    zone_mapping = data[['pickup_zone_enc', 'pickup_zone']].drop_duplicates().set_index('pickup_zone_enc')['pickup_zone'].to_dict()
else:
    # Fallback if the column names differ: adjust 'pickup_zone' to your actual raw string column name
    zone_mapping = {}


zone_demand_df = df_hotspot[['hour_bucket', 'pickup_zone_enc', 'ride_demand', 'hotspot']].copy()
if 'zone_mapping' in locals() and zone_mapping:
    zone_demand_df['pickup_zone_name'] = zone_demand_df['pickup_zone_enc'].map(zone_mapping)
zone_demand_df = zone_demand_df.sort_values('hour_bucket')

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
=== High Demand Hotspots at 18:00 ===
Guindy → High Demand Hotspot
Porur → High Demand Hotspot
Besant_Nagar → High Demand Hotspot
Adyar → High Demand Hotspot
Mambalam → High Demand Hotspot
Egmore → High Demand Hotspot
Anna_Nagar → High Demand Hotspot
Velachery → High Demand Hotspot
Saidapet → High Demand Hotspot
Tambaram → High Demand Hotspot


# save

In [ ]:
import os
import joblib
import tensorflow as tf

models_dir = '/content/drive/MyDrive/rideflow_AI/models'
os.makedirs(models_dir, exist_ok=True)


# 1. SAVE ETA MODEL & ARTIFACTS (LSTM)
eta_model.save(os.path.join(models_dir, 'eta_model.keras'))
eta_artifact = {
    "scaler_x": scaler_x,
    "scaler_y": scaler_y,
    "features": eta_features,
    'bins' : {'Low': 0, 'Medium': 1, 'High': 2}
}
joblib.dump(eta_artifact, os.path.join(models_dir, 'eta_artifact.pkl'))


# 2. SAVE HOTSPOT MODEL & ARTIFACTS
hotspot_model.save(os.path.join(models_dir, 'hotspot_model.keras'))

hotspot_artifact = {
    "scaler": scaler_hotspot,
    "features": hotspot_features,
    "zone_mapping": zone_mapping if 'zone_mapping' in locals() else {}
}
joblib.dump(hotspot_artifact, os.path.join(models_dir, 'hotspot_artifact.pkl'))

# 3. SAVE DRIVER BEHAVIOR CNN MODEL
cnn_model.save(os.path.join(models_dir, 'driver_behavior_cnn.keras'))

cnn_artifact = {
    "class_indices": train_data.class_indices if 'train_data' in locals() else {'Phone_Usage': 0, 'Rash_Driving': 1, 'Safe': 2}
}
joblib.dump(cnn_artifact, os.path.join(models_dir, 'cnn_artifact.pkl'))

print(" models and artifacts successfully saved ")

 models and artifacts successfully saved 


In [ ]:
# 1. Save ETA Dataset
eta_csv_path = '/content/drive/MyDrive/rideflow_AI/processed_data/eta_data.csv'
df_eta.to_csv(eta_csv_path, index=False)

# 2. Save Hotspot Dataset
hotspot_csv_path = '/content/drive/MyDrive/rideflow_AI/processed_data/hotspot_data.csv'
df_hotspot.to_csv(hotspot_csv_path, index=False)

#3. save zone wise demand hotspot
zone_demand_path = '/content/drive/MyDrive/rideflow_AI/processed_data/zone_demand_hotspot_data.csv'
zone_demand_df.to_csv(zone_demand_path, index=False)
print(" all dataset Saved ")

 all dataset Saved 


In [ ]:
file_path = '/content/drive/MyDrive/rideflow_AI/models/eta_features.pkl'
with open(file_path, "wb") as file:
    pickle.dump(eta_features, file)

file_path1 = '/content/drive/MyDrive/rideflow_AI/models/hotspot_features.pkl'
with open(file_path1, "wb") as file:
    pickle.dump(hotspot_features, file)

hotspot_enc = {'Low': 0, 'Medium': 1, 'High': 2}
file_path2 = '/content/drive/MyDrive/rideflow_AI/models/hotspot_enc.pkl'
with open(file_path2, "wb") as file:
    pickle.dump(hotspot_enc, file)

file_path3 = '/content/drive/MyDrive/rideflow_AI/models/eta_scale_x.pkl'
with open("file_path3.pkl","wb") as f:
    pickle.dump(scaler_x,f)

file_path4 = '/content/drive/MyDrive/rideflow_AI/models/eta_scale_y.pkl'
with open("file_path3.pkl","wb") as f:
    pickle.dump(scaler_y,f)

file_path4 = '/content/drive/MyDrive/rideflow_AI/models/scaler_hotspot.pkl'
with open("file_path4.pkl","wb") as f:
    pickle.dump(scaler_hotspot,f)

